# Data Collection
For this project, it is necessary to gather all the nba games played along with their box scores, team, and player performance. To gather, process, and generate this dataset in a local format, we can pull the html from a website which contains the information.

- Pull data from databases, APIs, logs, sensors, scrape, or label datasets
- Combine multiple data sources
- Capture metadata (source, timestamps, lineage)

Thankfully, www.basketball-reference.com has all the information we need to generate this dataset. Recording all the data by hand would be very time consuming. A method to pull all the data from the website is to use a bot to go through every relevant web page on the website and copy the html code containing the information we are looking for. We can then use pre-defined classes in the html code to extract key information.

Here we install the necessary libraries for running asyncronous web scraping.

In [ ]:
%pip install beautifulsoup4
%pip install playwright
# installs playwright for different web browsers using the command line '!'
!playwright install

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Since playwright opens a browser on the side (which takes time), playwright runs asyncronously. This means that functions are run on another thread and will return when the function is complete. Jupyter notebook on Windows is unable to process asyncronous playwright's event loop as JN is running it's own event loop which clashes.

To counteract this error, we can run a python script from jupyter notebook instead.

Website's are not a fan of scraper bots as they can take up a large chunk of a website's resources and generally cause issues. To avoid disturbing the website, our bot has to wait before trying to process a web page. If the bot were to try and rapidly process many pages, the website would kick the bot out and potentially ban them, ruining our project.  

Scraping the website for each game can potentially take hours due to the time waited between each request. This process is best run overnight.

The following script "scraper.py" has the main functions to go through the website's html pages and pull and download the html pages to our storage. Further functions can be used to process these html pages and pull important data from the page's tables into a pandas dataframe.

```
import os
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeout
from bs4 import BeautifulSoup
import time

# Seasons to scrape
SEASONS = list(range(2016, 2026))

# Directories to save scraped data
DATA_DIR = "data"
STANDINGS_DIR = os.path.join(DATA_DIR, "standings")
SCORES_DIR = os.path.join(DATA_DIR, "scores")

os.makedirs(STANDINGS_DIR, exist_ok=True)
os.makedirs(SCORES_DIR, exist_ok=True)

async def get_html(url, selector, sleep=5, retries=3):
    html = None
    for i in range(1, retries+1):
        time.sleep(sleep * i)
        try:
            async with async_playwright() as p:
                browser = await p.chromium.launch()
                page = await browser.new_page()
                await page.goto(url)
                print(await page.title())
                html = await page.inner_html(selector)
        except PlaywrightTimeout:
            print(f"Timeout error on {url}")
            continue
        else:
            break
    return html


async def scrape_season(season):
    url = f"https://www.basketball-reference.com/leagues/NBA_{season}_games.html"
    html = await get_html(url, "#content .filter")
    
    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    standings_pages = [f"https://www.basketball-reference.com{l['href']}" for l in links]
    
    for url in standings_pages:
        save_path = os.path.join(STANDINGS_DIR, url.split("/")[-1])
        if os.path.exists(save_path):
            continue
        
        html = await get_html(url, "#all_schedule")
        with open(save_path, "w+") as f:
            f.write(html)

async def scrape_game(standings_file):
    with open(standings_file, 'r') as f:
        html = f.read()
        
    soup = BeautifulSoup(html)
    links = soup.find_all("a")
    hrefs = [l.get("href") for l in links]
    box_scores = [l for l in hrefs if l and "boxscore" in l and ".html" in l]
    box_scores = [f"https://www.basketball-reference.com{l}" for l in box_scores]
    
    for url in box_scores:
        save_path = os.path.join(SCORES_DIR, url.split("/")[-1])
        if os.path.exists(save_path):
            continue
            
        html = await get_html(url, "#content")
        if not html:
            continue
        with open(save_path, "w+", encoding="utf-8") as f:
            f.write(html)
            
def scrape_seasons(seasons):
    for season in seasons:
        asyncio.run(scrape_season(season))
        
def scrape_all_games():
    standings_files = os.listdir(STANDINGS_DIR)
    standings_files = [s for s in standings_files if ".html" in s]
    
    for f in standings_files:
        filepath = os.path.join(STANDINGS_DIR, f)
        asyncio.run(scrape_game(filepath))

if __name__ == "__main__":
    scrape_seasons(SEASONS)
    scrape_all_games()
```

In [5]:
!python scraper.py

2015-16 NBA Schedule | Basketball-Reference.com
2016-17 NBA Schedule | Basketball-Reference.com
2017-18 NBA Schedule | Basketball-Reference.com
2018-19 NBA Schedule | Basketball-Reference.com
2019-20 NBA Schedule | Basketball-Reference.com
2020-21 NBA Schedule | Basketball-Reference.com
2021-22 NBA Schedule | Basketball-Reference.com
2022-23 NBA Schedule | Basketball-Reference.com
2023-24 NBA Schedule | Basketball-Reference.com
2024-25 NBA Schedule | Basketball-Reference.com


E:\MyNotebooks\NBA-Predictor\scraper.py:41: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 41 of the file E:\MyNotebooks\NBA-Predictor\scraper.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  soup = BeautifulSoup(html)
E:\MyNotebooks\NBA-Predictor\scraper.py:58: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 58 of the file E:\MyNotebooks\N

To save having to upload each html file to github, I move the scraped html files to my personal drive, and only intend to upload the raw csv file as a backup.

## Reading html files

Here we use the beautiful soup library to read the html files and grab the relevant information we are looking for. We can do this by looking for certain classes in the html code.

Not all games are required to have been scraped from the website before running this next section of code

### hmtltocsvconverter.py
```
import os
import pandas as pd
from bs4 import BeautifulSoup

SCORE_DIR = "data/scores"

box_scores = os.listdir(SCORE_DIR)
box_scores = [os.path.join(SCORE_DIR, f) for f in box_scores if f.endswith(".html")]

print(len(box_scores))

def parse_html(box_score):
    html = None
    # Try UTF-8 first
    try:
        with open(box_score, 'r', encoding="utf-8") as f:
            html = f.read()
    except UnicodeDecodeError:
        # Try cp1252 (Windows-1252) if UTF-8 fails
        try:
            with open(box_score, 'r', encoding="cp1252") as f:
                html = f.read()
        except UnicodeDecodeError:
            print(f"⚠️ Skipping file due to encoding error: {box_score}")
            return None

    if not html or html.strip() == "":
        print(f"⚠️ Skipping empty file: {box_score}")
        return None

    soup = BeautifulSoup(html, "html.parser")
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thead")]
    return soup

def read_line_score(soup):
    try:
        line_score = pd.read_html(str(soup), attrs={"id": "line_score"})[0]
    except (ValueError, IndexError):
        # ValueError: no tables found, IndexError: empty result
        return None

    cols = list(line_score.columns)
    cols[0] = "team"
    cols[-1] = "total"
    line_score.columns = cols

    return line_score[["team", "total"]]

def read_stats(soup, team, stat):
    try:
        df = pd.read_html(
            str(soup), attrs={"id": f"box-{team}-game-{stat}"}, index_col=0
        )[0]
    except (ValueError, IndexError):
        return None

    return df.apply(pd.to_numeric, errors="coerce")

def read_season_info(soup):
    try:
        nav = soup.select("#bottom_nav_container")[0]
        hrefs = [a["href"] for a in nav.find_all("a")]
        season = os.path.basename(hrefs[1]).split("_")[0]
        return season
    except Exception:
        return None

def main():
    base_cols = None
    games = []

    for box_score in box_scores:
        soup = parse_html(box_score)
        if soup is None:
            continue

        line_score = read_line_score(soup)
        if line_score is None:
            print(f"⚠️ Skipping file with no line_score: {box_score}")
            continue

        teams = list(line_score["team"])
        summaries = []

        for team in teams:
            basic = read_stats(soup, team, "basic")       #pulling the team's basic stats 
            advanced = read_stats(soup, team, "advanced") #pulling the team's more advanced stats
            if basic is None or advanced is None:
                print(f"⚠️ Skipping {box_score} due to missing stats for {team}")
                summaries = []
                break

            totals = pd.concat([basic.iloc[-1,:], advanced.iloc[-1,:]])
            totals.index = totals.index.str.lower()

            #Maximum statistic put up by one of the players
            #e.g. "GSW had a player put up the most points with 40pts, therefore pts_max=40"
            maxes = pd.concat([basic.iloc[:-1,:].max(), advanced.iloc[:-1,:].max()])
            maxes.index = maxes.index.str.lower() + "_max"

            summary = pd.concat([totals, maxes])

            if base_cols is None:
                base_cols = list(summary.index.drop_duplicates(keep="first"))
                base_cols = [b for b in base_cols if "bpm" not in b]

            summary = summary[base_cols]
            summaries.append(summary)

        if not summaries:
            continue  # skip incomplete game

        summary = pd.concat(summaries, axis=1).T
        game = pd.concat([summary, line_score], axis=1)

        game["home"] = [0, 1]
        game_opp = game.iloc[::-1].reset_index()
        game_opp.columns += "_opp"

        full_game = pd.concat([game, game_opp], axis=1)
        full_game["season"] = read_season_info(soup)
        full_game["date"] = pd.to_datetime(os.path.basename(box_score)[:8], format="%Y%m%d")
        full_game["won"] = full_game["total"] > full_game["total_opp"]

        games.append(full_game)

        if len(games) % 100 == 0:
            print(f"{len(games)} / {len(box_scores)}")

    games_df = pd.concat(games, ignore_index=True)
    games_df.to_csv("data/nba_games_raw.csv")

if __name__ == "__main__":
    main()
```